# 01 · What a tensor is / Qué es un tensor

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/01-what-a-tensor-is.ipynb)

*Part I · demo · 20 min*

A tensor is a way to **organize numbers using one or more directions, called axes**. In this notebook you will learn to read those axes before doing more advanced operations.

> 🇪🇸 Un tensor es una forma de **organizar números usando una o más direcciones, llamadas ejes**. En este cuaderno aprenderás a leer esos ejes antes de realizar operaciones más avanzadas.

## What you will be able to do / Lo que podrás hacer

- Explain **tensor, axis, order, shape, slice, fiber, unfolding, and contraction** in plain language.
- Read `.shape`, `.ndim`, and `.size` and say what the numbers mean.
- Follow what happens to the axes when data is selected, rearranged, or summed.
- Use simple `np.einsum` examples without treating the notation as a black box.

> 🇪🇸
>
> - Explicar en lenguaje sencillo **tensor, eje, orden, forma, corte, fibra, unfolding y contracción**.
> - Leer `.shape`, `.ndim` y `.size` y explicar qué significan sus números.
> - Seguir qué ocurre con los ejes cuando los datos se seleccionan, reorganizan o suman.
> - Usar ejemplos sencillos de `np.einsum` sin tratar la notación como una “caja negra”.

## Setup / Preparación

Run this cell first. It imports the small set of tools and real datasets used in this notebook.

> 🇪🇸 Ejecuta primero esta celda. Importa las herramientas y los conjuntos de datos reales que usaremos en este cuaderno.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from skimage import data

rng = np.random.default_rng(0)

print("EN: Setup ready.")
print("ES: Preparación lista.")

## Start with an everyday idea / Empecemos con una idea cotidiana

Think about how information can be arranged:

| Example / Ejemplo | What it looks like / Cómo se ve | Axes / Ejes |
|---|---|---:|
| One temperature / Una temperatura | `24.5` | 0 |
| Daily temperatures / Temperaturas diarias | `[21, 23, 24, ...]` | 1 |
| Excel-like table / Tabla tipo Excel | rows × columns / filas × columnas | 2 |
| Colour image / Imagen a color | height × width × colour / alto × ancho × color | 3 |
| Video / Video | time × height × width × colour / tiempo × alto × ancho × color | 4 |

The word **tensor** is the general name we use for these numerical objects when they can have any number of axes.

**Simple idea:** a tensor is not mysterious — it is numbers arranged in a structured way.

> 🇪🇸 La palabra **tensor** es el nombre general que usamos para estos objetos numéricos cuando pueden tener cualquier cantidad de ejes.
>
> **Idea sencilla:** un tensor no es algo misterioso; son números organizados de una manera estructurada.

## Why this matters / Por qué esto importa

In machine learning, many mistakes are not caused by a difficult formula. They happen because one axis is misunderstood.

Imagine a colour image with shape:

`(512, 512, 3)`

The `3` means **three colour channels**, not three images.

Now imagine a stack of digit images:

`(1797, 8, 8)`

The first number means **1,797 different images**.

Both objects have three axes, but those axes mean different things.

**Rule for the whole notebook:** before changing a tensor, ask:

1. What does each axis count?
2. Which axes will remain?
3. Which axes will disappear or move?

> 🇪🇸 En aprendizaje automático, muchos errores no ocurren por una fórmula difícil. Ocurren porque se interpreta mal un eje.
>
> Una imagen a color con forma `(512, 512, 3)` tiene tres canales de color; el `3` no significa tres imágenes.
>
> En cambio, `(1797, 8, 8)` puede significar 1.797 imágenes distintas de 8 × 8 píxeles.
>
> **Regla para todo el cuaderno:** antes de modificar un tensor, pregunta:
>
> 1. ¿Qué cuenta cada eje?
> 2. ¿Qué ejes permanecerán?
> 3. ¿Qué ejes desaparecerán o cambiarán de posición?

## 1.1 Vocabulary / Vocabulario

You do not need to memorize this table now. Use it as a dictionary while you work.

> 🇪🇸 No necesitas memorizar esta tabla ahora. Úsala como un diccionario mientras trabajas.

| Term / Término | Plain meaning / Significado sencillo | Example / Ejemplo |
|---|---|---|
| **Tensor / Tensor** | Numbers arranged along one or more axes / Números organizados a lo largo de uno o más ejes | A colour image / Una imagen a color |
| **Axis / Eje** | One direction in the data / Una dirección de los datos | height, width, colour / alto, ancho, color |
| **Mode / Modo** | Another name for an axis in tensor theory / Otro nombre para eje en teoría tensorial | mode 0 / modo 0 |
| **Order / Orden** | How many axes the tensor has / Cuántos ejes tiene el tensor | a matrix has order 2 / una matriz tiene orden 2 |
| **Shape / Forma** | The size of every axis / El tamaño de cada eje | `(512, 512, 3)` |
| **Slice / Corte** | Keep a “sheet” by fixing one index / Conservar una “lámina” fijando un índice | one colour channel / un canal de color |
| **Fiber / Fibra** | Keep one line of values by fixing all other indices / Conservar una línea de valores fijando los demás índices | RGB values of one pixel / valores RGB de un píxel |
| **Unfolding / Desplegado** | Rearrange a tensor into a matrix without throwing values away / Reorganizar un tensor como matriz sin eliminar valores | image tensor → matrix |
| **Contraction / Contracción** | Combine values by multiplying and summing over an axis / Combinar valores multiplicando y sumando sobre un eje | dot product / producto escalar |
| **Decomposition / Descomposición** | Rewrite data using simpler components / Reescribir los datos usando componentes más simples | SVD, Tucker, CP |

### One picture to remember / Una imagen mental para recordar

Think of a tensor as a **box of organized numbers**:

- `shape` tells you the box dimensions;
- `axis` tells you one direction inside the box;
- `order` counts how many directions there are.

> 🇪🇸 Piensa en un tensor como una **caja de números organizados**:
>
> - `shape` indica las dimensiones de la caja;
> - `axis` indica una dirección dentro de la caja;
> - `order` cuenta cuántas direcciones existen.

### Order vs. rank / Orden vs. rango

These words sound similar but mean different things.

**Order** is easy to read from the shape: it is simply the **number of axes**.

For example:

`shape = (32, 8, 8)`

has three numbers, so the tensor has **order 3**.

**Rank** is a different mathematical idea. For matrices, rank describes how many independent directions of information the matrix contains. Tensor rank is more advanced and has its own definitions.

**Do not use “order” and “rank” as synonyms.**

> 🇪🇸 Estas palabras pueden confundirse, pero significan cosas distintas.
>
> **Orden** es simplemente el **número de ejes**. Por ejemplo, `(32, 8, 8)` tiene tres ejes, así que es de **orden 3**.
>
> **Rango** es otra idea matemática. En matrices describe cuántas direcciones independientes de información contiene la matriz. El rango tensorial es más avanzado y tiene sus propias definiciones.
>
> **No uses “orden” y “rango” como sinónimos.**

<details>
<summary><strong>Quick check / Comprobación rápida</strong></summary>

**Question / Pregunta:** What is the order of `(32, 8, 8)`?

**Answer / Respuesta:** `3`, because there are three axes.

**EN:** The rank cannot be determined from the shape alone.

**ES:** El rango no puede determinarse únicamente a partir de la forma.

</details>

## 1.2 Shape in NumPy / Forma en NumPy

NumPy gives us three useful attributes:

- `.shape` → size of each axis / tamaño de cada eje
- `.ndim` → number of axes / número de ejes
- `.size` → total number of stored values / número total de valores

**Example / Ejemplo**

For shape `(2, 3, 4)`:

- order / orden = `3`
- size / tamaño total = `2 × 3 × 4 = 24`

> 🇪🇸 `.shape` responde “¿cómo están organizados los datos?”, `.ndim` responde “¿cuántos ejes hay?” y `.size` responde “¿cuántos valores hay en total?”.

In [ ]:
scalar = np.array(3.0)
vector = np.array([1., 2., 3.])
matrix = np.array([[1., 2.], [3., 4.]])
tensor = rng.standard_normal((2, 3, 4))

for name_en, name_es, arr in [
    ("scalar", "escalar", scalar),
    ("vector", "vector", vector),
    ("matrix", "matriz", matrix),
    ("order-3 tensor", "tensor de orden 3", tensor),
]:
    print(
        f"{name_en} / {name_es}: "
        f"shape/forma={arr.shape}, ndim/ejes={arr.ndim}, size/valores={arr.size}"
    )

### Why are these first arrays synthetic? / ¿Por qué estos primeros arreglos son sintéticos?

Here synthetic data is useful because the goal is only to see the pattern:

**0 axes → 1 axis → 2 axes → 3 axes**

There is no domain story to distract us. Immediately after this small example, we switch to real image data.

> 🇪🇸 Aquí los datos sintéticos son útiles porque queremos observar únicamente el patrón:
>
> **0 ejes → 1 eje → 2 ejes → 3 ejes**
>
> No hay detalles del mundo real que distraigan. Inmediatamente después pasamos a datos reales de imágenes.

### Same order, different meaning / Mismo orden, distinto significado

Now compare two **real** datasets:

- handwritten digits: `(1797, 8, 8)`
- colour photograph: `(512, 512, 3)`

Both have three axes.

But:

`(1797, 8, 8)` = **image × height × width**

while:

`(512, 512, 3)` = **height × width × colour**

That is why shape alone does not tell you the meaning.

> 🇪🇸 Ahora compara dos conjuntos **reales**:
>
> `(1797, 8, 8)` = **imagen × alto × ancho**
>
> `(512, 512, 3)` = **alto × ancho × color**
>
> Ambos tienen tres ejes, pero representan cosas diferentes. Por eso la forma por sí sola no contiene el significado.

In [ ]:
digits = load_digits()
photo = data.immunohistochemistry()

# D is the real handwritten-digit image tensor (image × row × column),
# loaded here in a visible cell so Exercise 3 and its explorer run whether
# or not the folded solutions are executed.
D = digits.images

print("Digits / Dígitos:", digits.images.shape)
print("EN: image × height × width")
print("ES: imagen × alto × ancho")
print()

print("Photo / Fotografía:", photo.shape)
print("EN: height × width × colour")
print("ES: alto × ancho × color")

### Interactive shape explorer / Explorador interactivo de formas

Choose an object and read its `shape`, `ndim`, and `size` as a sentence.

> 🇪🇸 Elige un objeto y lee su `shape`, `ndim` y `size` como una frase.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

shape_choice = widgets.Dropdown(
    options=[
        ("Scalar / Escalar", "scalar"),
        ("Vector / Vector", "vector"),
        ("Matrix / Matriz", "matrix"),
        ("Order-3 tensor / Tensor de orden 3", "tensor"),
        ("Real digit batch / Lote real de dígitos", "digits"),
        ("Real RGB photo / Foto RGB real", "photo"),
    ],
    value="digits",
    description="Object / Objeto:",
    style={"description_width": "120px"},
)

def explain_shape(choice):
    objects = {
        "scalar": ("scalar / escalar", scalar, "one number / un número"),
        "vector": ("vector / vector", vector, "one axis / un eje"),
        "matrix": ("matrix / matriz", matrix, "rows × columns / filas × columnas"),
        "tensor": (
            "order-3 tensor / tensor de orden 3",
            tensor,
            "axis 0 × axis 1 × axis 2 / eje 0 × eje 1 × eje 2",
        ),
        "digits": (
            "real digit batch / lote real de dígitos",
            digits.images,
            "images × height × width / imágenes × alto × ancho",
        ),
        "photo": (
            "real RGB photo / foto RGB real",
            photo,
            "height × width × colour / alto × ancho × color",
        ),
    }

    name, arr, meaning = objects[choice]

    print("Object / Objeto:", name)
    print("Shape / Forma:", arr.shape)
    print("ndim / número de ejes:", arr.ndim)
    print("size / número de valores:", arr.size)
    print("EN:", meaning.split(" / ")[0])
    print("ES:", meaning.split(" / ")[1])

shape_output = widgets.interactive_output(
    explain_shape,
    {"choice": shape_choice},
)

display(widgets.VBox([shape_choice, shape_output]))

## Exercise 1 — read shapes as sentences / Ejercicio 1 — lee las formas como frases

Do not answer with numbers only. Translate every shape into words.

**Example / Ejemplo**

`(512, 512, 3)` →

**EN:** 512 pixels high × 512 pixels wide × 3 colour channels.

**ES:** 512 píxeles de alto × 512 píxeles de ancho × 3 canales de color.

> 🇪🇸 En este ejercicio no basta con imprimir números. Debes explicar qué significa cada eje.

In [ ]:
# TODO 1 / TAREA 1
# EN: Build a scalar, vector, matrix, and order-3 tensor.
#     Print .shape, .ndim, and .size for each.
#     Which one has shape ()?
# ES: Construye un escalar, vector, matriz y tensor de orden 3.
#     Imprime .shape, .ndim y .size para cada uno.
#     ¿Cuál tiene shape ()?
#
# TODO 2 / TAREA 2
# EN: Inspect load_digits().images and data.astronaut().
#     For each dataset, explain what axis 0, axis 1, and axis 2 count.
# ES: Inspecciona load_digits().images y data.astronaut().
#     Para cada conjunto, explica qué cuentan los ejes 0, 1 y 2.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

examples = [
    ("scalar / escalar", np.array(3.0)),
    ("vector / vector", np.zeros(3)),
    ("matrix / matriz", np.zeros((2, 2))),
    ("order-3 tensor / tensor de orden 3", np.zeros((2, 3, 4))),
]

for name, arr in examples:
    print(f"{name}: shape={arr.shape}, ndim={arr.ndim}, size={arr.size}")

print()
D = load_digits().images
A = data.astronaut()

print("Digits / Dígitos:", D.shape)
print("EN: axis 0 = image, axis 1 = pixel row, axis 2 = pixel column.")
print("ES: eje 0 = imagen, eje 1 = fila de píxeles, eje 2 = columna de píxeles.")
print()

print("Astronaut / Astronauta:", A.shape)
print("EN: axis 0 = height, axis 1 = width, axis 2 = RGB colour channel.")
print("ES: eje 0 = alto, eje 1 = ancho, eje 2 = canal de color RGB.")

<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

The first task shows that `.ndim` is just the number of axes:

- scalar → 0
- vector → 1
- matrix → 2
- order-3 tensor → 3

The second task shows the more important lesson: **an axis gets its meaning from the data, not from its position alone**.

> 🇪🇸 La primera tarea muestra que `.ndim` es simplemente el número de ejes.
>
> La segunda muestra la idea más importante: **el significado de un eje proviene de los datos, no solamente de su posición**.
>
> Un eje 0 puede significar “qué imagen” en un conjunto y “alto de la imagen” en otro.

</details>

## 1.3 Three operations that matter / Tres operaciones importantes

We will now use three operations. Do not memorize formulas; remember the picture.

### 1) Slice and fiber / Corte y fibra

Imagine a loaf of sliced bread.

- A **slice** is like taking one whole slice from the loaf.
- A **fiber** is more like taking one thin line through the loaf.

For an RGB image:

`photo[:, :, 0]`

fixes the colour axis to channel 0 and keeps height and width → a 2D image.

`photo[100, 200, :]`

fixes one row and one column, but keeps the colour axis → three RGB values.

> 🇪🇸 Imagina una barra de pan cortada:
>
> - un **corte (slice)** conserva una “lámina” completa;
> - una **fibra (fiber)** conserva una línea de valores.
>
> En una imagen RGB, `photo[:, :, 0]` conserva una imagen 2D de un solo canal, mientras `photo[100, 200, :]` conserva los tres valores de color de un píxel.

In [ ]:
slice_red = photo[:, :, 0]
pixel_fiber = photo[100, 200, :]

print("Slice / Corte:", slice_red.shape)
print("EN: one colour channel, but height and width remain.")
print("ES: un canal de color; el alto y el ancho permanecen.")
print()

print("Fiber / Fibra:", pixel_fiber.shape)
print("EN: one pixel represented by its 3 colour values.")
print("ES: un píxel representado por sus 3 valores de color.")

### See slice and fiber together / Observa corte y fibra al mismo tiempo

Use the controls below to explore the image.

- **Row / Fila** and **Column / Columna** choose one pixel.
- **Channel / Canal** chooses which colour slice to display.
- The bar chart shows the selected pixel's three RGB values using the actual **red, green, and blue** colours.

Try moving to a bright red area, a green area, and a blue area. Watch how the three bars change.

> 🇪🇸 Usa los controles para explorar la imagen.
>
> - **Fila** y **Columna** eligen un píxel.
> - **Canal** selecciona qué corte de color mostrar.
> - El gráfico de barras muestra los tres valores RGB del píxel usando los colores reales **rojo, verde y azul**.
>
> Prueba diferentes zonas de la imagen y observa cómo cambian las tres barras.

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

channel_names = {
    0: ("Red", "Rojo", "R"),
    1: ("Green", "Verde", "G"),
    2: ("Blue", "Azul", "B"),
}

def show_slice_and_fiber(row, col, channel):
    plt.close("all")

    channel_en, channel_es, channel_letter = channel_names[channel]
    fiber = photo[row, col, :]

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))

    axes[0].imshow(photo)
    axes[0].scatter([col], [row], s=70)
    axes[0].set_title("Full image / Imagen completa")
    axes[0].axis("off")

    axes[1].imshow(photo[:, :, channel], cmap="gray")
    axes[1].scatter([col], [row], s=70)
    axes[1].set_title(
        f"{channel_en} slice / Corte {channel_es.lower()} ({channel_letter})"
    )
    axes[1].axis("off")

    axes[2].bar(
        ["R", "G", "B"],
        fiber,
        color=["red", "green", "blue"],
    )
    axes[2].set_title("Pixel fiber / Fibra del píxel")
    axes[2].set_ylim(0, 255)
    axes[2].set_ylabel("Intensity / Intensidad")

    plt.tight_layout()
    plt.show()

    print(
        f"EN: pixel at row={row}, column={col} has "
        f"R={fiber[0]}, G={fiber[1]}, B={fiber[2]}."
    )
    print(
        f"ES: el píxel en fila={row}, columna={col} tiene "
        f"R={fiber[0]}, G={fiber[1]}, B={fiber[2]}."
    )
    print(
        f"EN: the middle panel shows only the {channel_en.lower()} channel."
    )
    print(
        f"ES: el panel central muestra únicamente el canal {channel_es.lower()}."
    )

row_slider = widgets.IntSlider(
    min=0, max=photo.shape[0] - 1, step=1, value=100,
    description="Row / Fila:", continuous_update=False,
    style={"description_width": "110px"},
)

col_slider = widgets.IntSlider(
    min=0, max=photo.shape[1] - 1, step=1, value=200,
    description="Column / Columna:", continuous_update=False,
    style={"description_width": "110px"},
)

channel_selector = widgets.ToggleButtons(
    options=[
        ("R · Red / Rojo", 0),
        ("G · Green / Verde", 1),
        ("B · Blue / Azul", 2),
    ],
    value=0,
    description="Channel / Canal:",
    style={"description_width": "110px"},
)

slice_output = widgets.interactive_output(
    show_slice_and_fiber,
    {
        "row": row_slider,
        "col": col_slider,
        "channel": channel_selector,
    },
)

display(
    widgets.VBox([
        widgets.HBox([row_slider, col_slider]),
        channel_selector,
        slice_output,
    ])
)

### 2) Unfolding / Desplegado

Sometimes an algorithm expects a **matrix**, but our data has three or more axes.

Unfolding solves that formatting problem.

Think of taking a folded map and opening it flat on a table: **the information is still there; only the arrangement changes**.

For a colour image `(H, W, C)`, an unfolding keeps one axis separate and combines the others into one long axis.

No pixel value is deleted.

> 🇪🇸 Algunas herramientas esperan una **matriz**, pero nuestros datos pueden tener tres o más ejes.
>
> El unfolding resuelve ese problema de formato.
>
> Piensa en desplegar un mapa doblado sobre una mesa: **la información sigue allí; solamente cambia su organización**.
>
> No se elimina ningún píxel.

In [ ]:
def unfold(T, axis):
    # Move one axis to the front and flatten the others.
    return np.moveaxis(T, axis, 0).reshape(T.shape[axis], -1)

for axis in [0, 1, 2]:
    M = unfold(photo, axis)
    print(
        f"axis/eje {axis}: {photo.shape} -> {M.shape} | "
        f"same number of values / mismo número de valores: {M.size == photo.size}"
    )

### Interactive unfolding explorer / Explorador interactivo de unfolding

Change the axis and watch how the same image becomes a different matrix shape. No values are deleted.

> 🇪🇸 Cambia el eje y observa cómo la misma imagen se convierte en una matriz de forma diferente. No se elimina ningún valor.


In [ ]:
unfold_axis = widgets.ToggleButtons(
    options=[
        ("Axis 0 / Eje 0", 0),
        ("Axis 1 / Eje 1", 1),
        ("Axis 2 / Eje 2", 2),
    ],
    value=2,
    description="Axis / Eje:",
    style={"description_width": "90px"},
)

def explore_unfolding(axis):
    M = unfold(photo, axis)

    plt.close("all")
    fig, ax = plt.subplots(figsize=(8.5, 3.2))
    ax.imshow(M, aspect="auto", cmap="gray")
    ax.set_title(
        f"Unfold axis {axis} / Desplegado por eje {axis} — shape {M.shape}"
    )
    ax.set_xlabel("Flattened direction / Dirección aplanada")
    ax.set_ylabel(f"Kept axis {axis} / Eje {axis} conservado")
    plt.tight_layout()
    plt.show()

    print("Original shape / Forma original:", photo.shape)
    print("Matrix shape / Forma matricial:", M.shape)
    print(
        "Same number of values / Mismo número de valores:",
        M.size == photo.size,
    )
    print(
        "EN: unfolding changes arrangement, not information."
    )
    print(
        "ES: el unfolding cambia la organización, no la información."
    )

unfold_output = widgets.interactive_output(
    explore_unfolding,
    {"axis": unfold_axis},
)

display(widgets.VBox([unfold_axis, unfold_output]))

**What does the result mean? / ¿Qué significa el resultado?**

For `axis=2`, the image becomes a matrix with shape:

`(3, 262144)`

That means:

- 3 rows = red, green, blue;
- 262,144 columns = all spatial pixel positions flattened into one direction.

The operation changed the arrangement, not the stored information.

> 🇪🇸 Para `axis=2`, la imagen se convierte en una matriz `(3, 262144)`:
>
> - 3 filas = rojo, verde y azul;
> - 262.144 columnas = todas las posiciones espaciales de los píxeles reunidas en una sola dirección.
>
> Cambió la organización, no la información almacenada.

### 3) Contraction / Contracción

Contraction sounds technical, but the idea is familiar:

**multiply matching values and add them together.**

A dot product does exactly that.

For:

`[1, 2, 3] · [4, 5, 6]`

we calculate:

`1×4 + 2×5 + 3×6 = 32`

`np.einsum` gives us a notation that explicitly shows which index disappears because it is summed.

> 🇪🇸 “Contracción” suena técnico, pero la idea es conocida:
>
> **multiplicar valores correspondientes y después sumarlos.**
>
> El producto escalar hace exactamente eso. `np.einsum` permite mostrar explícitamente qué índice desaparece porque se suma.

### The `einsum` rule in one sentence / La regla de `einsum` en una frase

**If an index appears before `->` but disappears after `->`, that index is summed over.**

Example:

`i,i->`

`i` disappears → sum over `i` → one number.

For matrix multiplication:

`ik,kj->ij`

`k` disappears → sum over `k`.

`i` and `j` remain → the output is a matrix indexed by `i,j`.

> 🇪🇸 **Si un índice aparece antes de `->` pero desaparece después de `->`, ese índice se suma.**
>
> En `i,i->`, `i` desaparece y obtenemos un solo número.
>
> En `ik,kj->ij`, `k` desaparece porque se suma; `i` y `j` permanecen y forman la matriz de salida.

In [ ]:
a = np.array([1., 2., 3.])
b = np.array([4., 5., 6.])

dot_einsum = np.einsum("i,i->", a, b)
dot_numpy = np.dot(a, b)

print("Dot product / Producto escalar")
print("einsum:", dot_einsum, "| np.dot:", dot_numpy)
print("EN: index i disappears, so it is summed.")
print("ES: el índice i desaparece, por lo tanto se suma.")
print()

A = np.array([[1., 2.], [3., 4.]])
B = np.array([[5., 6.], [7., 8.]])

mat_einsum = np.einsum("ik,kj->ij", A, B)
mat_numpy = A @ B

print("Matrix product / Producto matricial")
print(mat_einsum)
print("Same as @ / Igual que @:", np.allclose(mat_einsum, mat_numpy))
print("EN: k disappears; i and j remain.")
print("ES: k desaparece; i y j permanecen.")

### Interactive contraction explorer / Explorador interactivo de contracción

Move the sliders and watch the dot product change. The operation is always the same: multiply matching positions, then add them.

> 🇪🇸 Mueve los controles y observa cómo cambia el producto escalar. La operación siempre es la misma: multiplicar posiciones correspondientes y después sumarlas.


In [ ]:
a1 = widgets.IntSlider(value=1, min=-5, max=5, description="a₁")
a2 = widgets.IntSlider(value=2, min=-5, max=5, description="a₂")
a3 = widgets.IntSlider(value=3, min=-5, max=5, description="a₃")
b1 = widgets.IntSlider(value=4, min=-5, max=5, description="b₁")
b2 = widgets.IntSlider(value=5, min=-5, max=5, description="b₂")
b3 = widgets.IntSlider(value=6, min=-5, max=5, description="b₃")

for slider in [a1, a2, a3, b1, b2, b3]:
    slider.continuous_update = False

def explore_dot(a1, a2, a3, b1, b2, b3):
    va = np.array([a1, a2, a3], dtype=float)
    vb = np.array([b1, b2, b3], dtype=float)
    products = va * vb
    result = np.einsum("i,i->", va, vb)

    print("a =", va.tolist())
    print("b =", vb.tolist())
    print("Pairwise products / Productos por posición:", products.tolist())
    print("Dot product / Producto escalar:", result)
    print(
        "EN:",
        f"{a1}×{b1} + {a2}×{b2} + {a3}×{b3} = {result:g}",
    )
    print(
        "ES:",
        f"{a1}×{b1} + {a2}×{b2} + {a3}×{b3} = {result:g}",
    )
    print("EN: index i disappears because its three positions are summed.")
    print("ES: el índice i desaparece porque sus tres posiciones se suman.")

dot_output = widgets.interactive_output(
    explore_dot,
    {
        "a1": a1, "a2": a2, "a3": a3,
        "b1": b1, "b2": b2, "b3": b3,
    },
)

display(
    widgets.VBox([
        widgets.HTML("<b>Vector a</b>"),
        widgets.HBox([a1, a2, a3]),
        widgets.HTML("<b>Vector b</b>"),
        widgets.HBox([b1, b2, b3]),
        dot_output,
    ])
)

## Exercise 2 — follow the axes / Ejercicio 2 — sigue los ejes

For every operation, answer two questions:

1. What happened to the axes?
2. What does the result mean in ordinary language?

> 🇪🇸 Para cada operación responde dos preguntas:
>
> 1. ¿Qué ocurrió con los ejes?
> 2. ¿Qué significa el resultado en lenguaje cotidiano?

In [ ]:
# TODO 3 / TAREA 3
# EN: From `photo`, extract:
#     a) the green-channel slice
#     b) the RGB fiber at pixel (10, 20)
#     Explain why one is a slice and the other is a fiber.
# ES: A partir de `photo`, extrae:
#     a) el corte del canal verde
#     b) la fibra RGB del píxel (10, 20)
#     Explica por qué uno es un corte y el otro una fibra.
#
# TODO 4 / TAREA 4
# EN: Unfold `photo` along axes 0, 1, and 2.
#     Confirm that every result contains exactly `photo.size` values.
# ES: Despliega `photo` por los ejes 0, 1 y 2.
#     Confirma que cada resultado contiene exactamente `photo.size` valores.
#
# TODO 5 / TAREA 5
# EN: Compute the dot product with einsum and verify it with np.dot.
#     Compute the matrix product with einsum and verify it with @.
# ES: Calcula el producto escalar con einsum y verifícalo con np.dot.
#     Calcula el producto matricial con einsum y verifícalo con @.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

green = photo[:, :, 1]
fiber = photo[10, 20, :]

print("Slice / Corte:", green.shape)
print("EN: colour is fixed; height and width remain.")
print("ES: el color se fija; el alto y el ancho permanecen.")
print()

print("Fiber / Fibra:", fiber.shape)
print("EN: row and column are fixed; the 3 colour values remain.")
print("ES: fila y columna se fijan; permanecen los 3 valores de color.")
print()

for axis in range(3):
    M = unfold(photo, axis)
    print(
        f"Unfold axis/eje {axis}: shape/forma={M.shape}, "
        f"keeps all values/conserva todos los valores={M.size == photo.size}"
    )

print()
print(
    "Dot product same / Producto escalar igual:",
    np.isclose(np.einsum("i,i->", a, b), np.dot(a, b)),
)
print(
    "Matrix product same / Producto matricial igual:",
    np.allclose(np.einsum("ik,kj->ij", A, B), A @ B),
)

<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

- **Slice:** one index is fixed and two axes remain.
- **Fiber:** two indices are fixed and one axis remains.
- **Unfolding:** axes are rearranged, but no values disappear.
- **Contraction:** selected axes disappear because their values are summed.

> 🇪🇸
>
> - **Corte:** se fija un índice y permanecen dos ejes.
> - **Fibra:** se fijan dos índices y permanece un eje.
> - **Unfolding:** los ejes se reorganizan, pero ningún valor desaparece.
> - **Contracción:** algunos ejes desaparecen porque sus valores se suman.

</details>

## Exercise 3 — Axis Reasoning Challenge / Ejercicio 3 — Reto de razonamiento sobre ejes

Let:

`D = load_digits().images`

with:

`D.shape = (1797, 8, 8)`

Read it as:

**images × height × width / imágenes × alto × ancho**

Before running each operation, predict the output shape.

| Operation / Operación | Question / Pregunta |
|---|---|
| `D[0]` | What happens when we choose one image? / ¿Qué ocurre al elegir una imagen? |
| `D[:, 3, 4]` | What remains when row and column are fixed? / ¿Qué queda al fijar fila y columna? |
| `unfold(D, 0)` | How many pixels does each image contain? / ¿Cuántos píxeles contiene cada imagen? |
| `np.einsum('nhw->n', D)` | Which axes disappear because they are summed? / ¿Qué ejes desaparecen porque se suman? |

> 🇪🇸 Primero predice. Después ejecuta. Finalmente explica el resultado con palabras.

In [ ]:
# TODO 6 / TAREA 6
# EN: Predict and then compute the shapes of:
#     1. D[0]
#     2. D[:, 3, 4]
#     3. unfold(D, 0)
#     4. np.einsum("nhw->n", D)
#
#     For each operation, explain which axes are fixed, kept,
#     rearranged, or summed.
#
# ES: Predice y después calcula las formas de:
#     1. D[0]
#     2. D[:, 3, 4]
#     3. unfold(D, 0)
#     4. np.einsum("nhw->n", D)
#
#     Para cada operación, explica qué ejes se fijan, conservan,
#     reorganizan o suman.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

D = load_digits().images

results = [
    ("D[0]", D[0]),
    ("D[:, 3, 4]", D[:, 3, 4]),
    ("unfold(D, 0)", unfold(D, 0)),
    ("einsum nhw->n", np.einsum("nhw->n", D)),
]

for name, result in results:
    print(f"{name:18s} -> shape/forma={result.shape}")

print()
print("1 EN: choose one image -> n disappears; h,w remain.")
print("1 ES: elegir una imagen -> n desaparece; h,w permanecen.")
print("2 EN: fix h and w -> n remains.")
print("2 ES: fijar h y w -> n permanece.")
print("3 EN: each 8×8 image becomes one row of 64 values.")
print("3 ES: cada imagen 8×8 se convierte en una fila de 64 valores.")
print("4 EN: h and w are summed; one total remains for each image n.")
print("4 ES: h y w se suman; queda un total para cada imagen n.")

### Interactive axis explorer / Explorador interactivo de ejes

Choose an operation below. The notebook will show the output shape and explain what happened to the axes.

> 🇪🇸 Elige una operación. El cuaderno mostrará la forma de salida y explicará qué ocurrió con los ejes.

In [ ]:
operation = widgets.Dropdown(
    options=[
        ("D[0] — one image / una imagen", "image"),
        ("D[:, 3, 4] — one pixel across images / un píxel en todas las imágenes", "fiber"),
        ("unfold(D, 0) — flatten each image / aplanar cada imagen", "unfold"),
        ("einsum nhw->n — sum pixels / sumar píxeles", "contract"),
    ],
    value="image",
    description="Operation / Operación:",
    style={"description_width": "150px"},
)

def explain_operation(choice):
    if choice == "image":
        result = D[0]
        en = "n is fixed to one image; h and w remain."
        es = "n se fija a una imagen; h y w permanecen."
    elif choice == "fiber":
        result = D[:, 3, 4]
        en = "h and w are fixed; n remains, so we get one pixel value per image."
        es = "h y w se fijan; n permanece, así obtenemos un valor de píxel por imagen."
    elif choice == "unfold":
        result = unfold(D, 0)
        en = "n remains as rows; each 8×8 image becomes 64 columns."
        es = "n permanece como filas; cada imagen 8×8 se convierte en 64 columnas."
    else:
        result = np.einsum("nhw->n", D)
        en = "h and w disappear because they are summed; n remains."
        es = "h y w desaparecen porque se suman; n permanece."

    print("Shape / Forma:", result.shape)
    print("EN:", en)
    print("ES:", es)

axis_output = widgets.interactive_output(
    explain_operation,
    {"choice": operation},
)

display(widgets.VBox([operation, axis_output]))

## 1.4 The factorization ladder / La escalera de factorizaciones

A **factorization** rewrites one object as a product — or a sum — of simpler pieces. Nothing is added and nothing is lost. The second form is chosen because it makes **one particular question** easy.

That is the ladder you just saw on the slide:

**number → polynomial → matrix → tensor**

Here you run it. Four rungs follow, each one object written a second way, each one printing the answer that the second way hands you for free.

**The question to keep asking:** *which question does this factorization make easy?*

> 🇪🇸 Una **factorización** reescribe un objeto como un producto —o una suma— de piezas más simples. No se añade ni se pierde nada. La segunda forma se elige porque vuelve fácil **una pregunta concreta**.
>
> Esa es la escalera que acabas de ver en la diapositiva:
>
> **número → polinomio → matriz → tensor**
>
> Aquí la ejecutas. Siguen cuatro peldaños: un objeto escrito de otra manera, y la respuesta que esa otra manera te regala.
>
> **La pregunta que hay que repetir:** *¿qué pregunta vuelve fácil esta factorización?*

### Rung 1 — a number / Peldaño 1 — un número

Take `100`. One object, four ways to write it, four different questions made trivial:

| Written as / Escrito como | Makes easy / Vuelve fácil |
|---|---|
| `2² × 5²` | how many divisors it has, and its gcd with another number / cuántos divisores tiene, y su mcd con otro número |
| `4 × 25` | multiplying it in your head / multiplicarlo mentalmente |
| `64 + 32 + 4` | how a computer stores it / cómo lo almacena un ordenador |
| `6² + 8²` | seeing the 6–8–10 right triangle / ver el triángulo rectángulo 6–8–10 |

The number never changed. Only the question did.

> 🇪🇸 Toma `100`. Un objeto, cuatro formas de escribirlo, cuatro preguntas distintas vueltas triviales.
>
> El número nunca cambió. Lo que cambió fue la pregunta.

In [ ]:
import math

n = 100

# 1. Prime factorization / Factorización en primos: 100 = 2² × 5².
#    The divisor count and the gcd are read straight off the exponents.
primes_100 = {2: 2, 5: 2}
primes_90 = {2: 1, 3: 2, 5: 1}          # 90 = 2 × 3² × 5

n_divisors = math.prod(e + 1 for e in primes_100.values())
shared = {p: min(e, primes_90[p]) for p, e in primes_100.items() if p in primes_90}
gcd_from_factors = math.prod(p ** e for p, e in shared.items())

print("100 = " + " × ".join(f"{p}^{e}" for p, e in primes_100.items()))
print("Divisors / Divisores:", n_divisors,
      "| counted one by one / contados uno a uno:",
      sum(1 for d in range(1, n + 1) if n % d == 0))
print("Shared prime powers / Potencias primas comunes:", shared)
print("gcd(100, 90) from the factors / desde los factores:", gcd_from_factors,
      "| math.gcd:", math.gcd(100, 90))
print()

# 2. Arithmetic / Aritmética.  3. Storage / Almacenamiento.  4. Geometry / Geometría.
print("4 × 25      =", 4 * 25, " -> EN: easy in your head / ES: fácil de cabeza")
print("64 + 32 + 4 =", 64 + 32 + 4, " -> EN/ES: bin(100) =", bin(n))
print("6² + 8²     =", 6 ** 2 + 8 ** 2,
      " -> EN: 10 is the hypotenuse of the 6-8-10 triangle"
      " / ES: 10 es la hipotenusa del triángulo 6-8-10")
print()
print("EN: one object, four factorizations, four different questions made trivial.")
print("ES: un objeto, cuatro factorizaciones, cuatro preguntas vueltas triviales.")

### Rung 2 — a quadratic / Peldaño 2 — una cuadrática

Same move, one rung up. `x² − 4x − 5` has three useful forms:

- **standard** `x² − 4x − 5` — easy to add and to differentiate;
- **factored** `(x − 5)(x + 1)` — the roots are read off: `x = 5` and `x = −1`;
- **completed square** `(x − 2)² − 9` — the vertex is read off: `(2, −9)`.

All three are the same function. Want the roots? Factor. Want the minimum? Complete the square.

**Worth saying out loud:** completing the square is the scalar version of **diagonalizing a quadratic form**. `(x − 2)² − 9` is what a quadratic looks like once its linear term has been absorbed; in many variables that same job is done by an eigendecomposition — a factorization of a *matrix*. The ladder is one idea, not four.

> 🇪🇸 El mismo movimiento, un peldaño más arriba. `x² − 4x − 5` tiene tres formas útiles: la **estándar**, fácil de sumar y derivar; la **factorizada** `(x − 5)(x + 1)`, donde las raíces se leen directamente; y el **cuadrado completado** `(x − 2)² − 9`, donde se lee el vértice `(2, −9)`.
>
> Las tres son la misma función. ¿Quieres las raíces? Factoriza. ¿Quieres el mínimo? Completa el cuadrado.
>
> **Conviene decirlo en voz alta:** completar el cuadrado es la versión escalar de **diagonalizar una forma cuadrática**. En varias variables ese mismo trabajo lo hace una descomposición espectral, que es una factorización de una *matriz*. La escalera es una sola idea, no cuatro.

In [ ]:
from numpy.polynomial import Polynomial

x = Polynomial([0.0, 1.0])          # the polynomial "x" / el polinomio «x»

standard = x ** 2 - 4 * x - 5
factored = (x - 5) * (x + 1)
completed = (x - 2) ** 2 - 9

print("Coefficients [c0, c1, c2] / Coeficientes:", standard.coef)
print("Factored is the same polynomial / La factorizada es el mismo polinomio:",
      np.allclose(standard.coef, factored.coef))
print("Completed square too / El cuadrado completado también:",
      np.allclose(standard.coef, completed.coef))
print()

vertex_x, vertex_y = 2.0, -9.0
print("(x - 5)(x + 1) -> roots / raíces:", standard.roots())
print(f"(x - 2)² - 9   -> vertex / vértice: ({vertex_x:g}, {vertex_y:g})",
      "| check / comprobación: f(2) =", standard(vertex_x))
print()
print("EN: roots from the factored form, vertex from the completed square.")
print("ES: las raíces desde la forma factorizada, el vértice desde el cuadrado completado.")

### Rung 3 — a fraction / Peldaño 3 — una fracción

This rung is not on the slide, and it is the clearest case of all: a factorization whose entire value is **downstream**.

`1 / (x² + x − 2)` is awkward to integrate. Factor the denominator, split the fraction, and the hard thing becomes two easy ones:

`1 / ((x − 1)(x + 2)) = (1/3)/(x − 1) − (1/3)/(x + 2)`

Each piece integrates to a logarithm on sight. The fraction did not get simpler — the *operation* did. (Engineers meet the same trick as the inverse Laplace transform: factor the denominator, and each piece inverts by inspection.)

> 🇪🇸 Este peldaño no está en la diapositiva, y es el caso más claro de todos: una factorización cuyo valor está **por completo en lo que viene después**.
>
> `1 / (x² + x − 2)` es incómoda de integrar. Factoriza el denominador, separa la fracción, y lo difícil se convierte en dos cosas fáciles:
>
> `1 / ((x − 1)(x + 2)) = (1/3)/(x − 1) − (1/3)/(x + 2)`
>
> Cada pieza se integra a un logaritmo a simple vista. La fracción no se simplificó; se simplificó la *operación*. (En ingeniería es el mismo truco de la transformada inversa de Laplace.)

In [ ]:
from scipy.integrate import quad

# The denominator factors: x² + x - 2 = (x - 1)(x + 2).
denominator = Polynomial([-2.0, 1.0, 1.0])
print("Roots of the denominator / Raíces del denominador:", denominator.roots())

# Residues by hand / Residuos a mano:
#   at x = 1:  1 / (1 + 2) =  1/3
#   at x = -2: 1 / (-2 - 1) = -1/3
res_a, res_b = 1 / 3, -1 / 3

original = lambda t: 1.0 / ((t - 1) * (t + 2))
split = lambda t: res_a / (t - 1) + res_b / (t + 2)

grid = np.linspace(3.0, 6.0, 7)
print("Same function on a grid / La misma función en una malla:",
      np.allclose(original(grid), split(grid)))
print()

# The payoff / La recompensa: the integral over [3, 6].
numeric, _ = quad(original, 3.0, 6.0)
antiderivative = lambda t: res_a * np.log(abs(t - 1)) + res_b * np.log(abs(t + 2))
from_split = antiderivative(6.0) - antiderivative(3.0)

print(f"Numeric integral / Integral numérica: {numeric:.12f}")
print(f"Two logarithms   / Dos logaritmos   : {from_split:.12f}")
print("Agree / Coinciden:", np.isclose(numeric, from_split))
print()
print("EN: the factorization did not simplify the fraction — it made integration easy.")
print("ES: la factorización no simplificó la fracción; volvió fácil la integración.")

### Rung 4 — a matrix / Peldaño 4 — una matriz

The fourth rung is the same move on an object with two axes — not a new topic.

- **LU:** `A = P L U`. Gaussian elimination, *saved*. Factor once, then solve `Ax = b` cheaply for as many right-hand sides as you like.
- **QR:** `A = Q R` with `Qᵀ Q = I`. Perpendicular, unit-length directions — the stable way to do least squares.

Notice what has not changed since rung 1: nothing is added, nothing is lost, and each form exists because it makes one question cheap.

> 🇪🇸 El cuarto peldaño es el mismo movimiento sobre un objeto con dos ejes; no es un tema nuevo.
>
> - **LU:** `A = P L U`. La eliminación gaussiana, *guardada*. Factoriza una vez y resuelve `Ax = b` barato para tantos lados derechos como quieras.
> - **QR:** `A = Q R` con `Qᵀ Q = I`. Direcciones perpendiculares de longitud 1: la forma estable de hacer mínimos cuadrados.
>
> Fíjate en lo que no ha cambiado desde el peldaño 1: no se añade nada, no se pierde nada, y cada forma existe porque vuelve barata una pregunta.

In [ ]:
from scipy.linalg import lu, lu_factor, lu_solve

A_ladder = np.array([[4.0, 3.0, 2.0],
                     [2.0, 1.0, 1.0],
                     [6.0, 3.0, 5.0]])

P, L, U = lu(A_ladder)
print("LU:  A = P L U ->", np.allclose(P @ L @ U, A_ladder))
print("L lower triangular / L triangular inferior:", np.allclose(L, np.tril(L)))
print("U upper triangular / U triangular superior:", np.allclose(U, np.triu(U)))
print()

Q, R = np.linalg.qr(A_ladder)
print("QR:  A = Q R ->", np.allclose(Q @ R, A_ladder))
print("Q orthonormal columns / columnas ortonormales:",
      np.allclose(Q.T @ Q, np.eye(3)))
print()

# Why bother / Para qué: factor once, solve many times.
ladder_factors = lu_factor(A_ladder)
for rhs in [np.array([1.0, 2.0, 3.0]), np.array([0.0, 1.0, 0.0])]:
    solution = lu_solve(ladder_factors, rhs)
    print(f"b = {rhs} -> x = {np.round(solution, 6)}",
          "| A @ x == b:", np.allclose(A_ladder @ solution, rhs))
print()
print("EN: one factorization, reused for every right-hand side.")
print("ES: una factorización, reutilizada para cada lado derecho.")

### Interactive ladder explorer / Explorador interactivo de la escalera

Choose a rung. The same three lines appear every time: the object, its factorization, and the question that factorization makes easy.

> 🇪🇸 Elige un peldaño. Siempre aparecen las mismas tres líneas: el objeto, su factorización y la pregunta que esa factorización vuelve fácil.

In [ ]:
rung = widgets.Dropdown(
    options=[
        ("1 · Number / Número — 100", "number"),
        ("2 · Quadratic / Cuadrática — x² - 4x - 5", "quadratic"),
        ("3 · Fraction / Fracción — 1/(x² + x - 2)", "fraction"),
        ("4 · Matrix / Matriz — 3 × 3", "matrix"),
    ],
    value="number",
    description="Rung / Peldaño:",
    style={"description_width": "130px"},
)

def explore_rung(choice):
    if choice == "number":
        rows = [
            ("Object / Objeto", "100"),
            ("Factorization / Factorización", "2² × 5²"),
            ("Makes easy / Vuelve fácil",
             "how many divisors / cuántos divisores: (2+1)(2+1) = 9"),
            ("Verified / Verificado",
             f"{math.prod(e + 1 for e in primes_100.values())} divisors, "
             f"gcd(100, 90) = {math.gcd(100, 90)}"),
        ]
    elif choice == "quadratic":
        rows = [
            ("Object / Objeto", "x² - 4x - 5"),
            ("Factorization / Factorización", "(x - 5)(x + 1)"),
            ("Makes easy / Vuelve fácil", "the roots / las raíces: x = 5, x = -1"),
            ("Verified / Verificado", f"roots / raíces = {standard.roots()}"),
        ]
    elif choice == "fraction":
        rows = [
            ("Object / Objeto", "1 / (x² + x - 2)"),
            ("Factorization / Factorización", "(1/3)/(x - 1) - (1/3)/(x + 2)"),
            ("Makes easy / Vuelve fácil",
             "the integral / la integral: two logarithms / dos logaritmos"),
            ("Verified / Verificado",
             f"∫ over / sobre [3, 6] = {numeric:.9f}"),
        ]
    else:
        rows = [
            ("Object / Objeto", f"A, shape / forma {A_ladder.shape}"),
            ("Factorization / Factorización", "A = P L U   and / y   A = Q R"),
            ("Makes easy / Vuelve fácil",
             "solving A x = b again and again / resolver A x = b una y otra vez"),
            ("Verified / Verificado",
             f"P L U == A: {np.allclose(P @ L @ U, A_ladder)}, "
             f"Q R == A: {np.allclose(Q @ R, A_ladder)}"),
        ]

    for label, value in rows:
        print(f"{label:32s} {value}")
    print()
    print("EN: same object, second form, one question made easy.")
    print("ES: el mismo objeto, una segunda forma, una pregunta vuelta fácil.")

rung_output = widgets.interactive_output(explore_rung, {"choice": rung})

display(widgets.VBox([rung, rung_output]))

## Exercise 4 — climb the ladder yourself / Ejercicio 4 — sube la escalera tú mismo

Two rungs, and the same discipline in both: get the answer **from the factorization**, then check it against something that computes the answer directly.

> 🇪🇸 Dos peldaños y la misma disciplina en ambos: obtén la respuesta **a partir de la factorización** y después compárala con algo que la calcule directamente.

In [ ]:
# TODO 7 / TAREA 7
# EN: 360 = 2³ × 3² × 5 and 84 = 2² × 3 × 7.
#     Read gcd(360, 84) off the shared prime powers — for each prime both
#     numbers have, keep the smaller exponent — and check it with math.gcd.
#     Then say how many divisors 360 has, from its exponents alone.
# ES: 360 = 2³ × 3² × 5 y 84 = 2² × 3 × 7.
#     Deduce mcd(360, 84) de las potencias primas comunes — de cada primo
#     compartido, conserva el exponente menor — y compruébalo con math.gcd.
#     Después di cuántos divisores tiene 360, solo con sus exponentes.
#
# TODO 8 / TAREA 8
# EN: Take x² - 6x + 8. Write its factored form and its completed square,
#     confirm all three forms have the same coefficients, and read the roots
#     off one and the vertex off the other.
#     Then LU-factor A_ex = [[2, 1, 1], [4, -6, 0], [-2, 7, 2]] once with
#     scipy.linalg.lu_factor, and solve A_ex @ x = b for b = [5, -2, 9] and
#     b = [1, 0, 0] reusing that same factorization.
# ES: Toma x² - 6x + 8. Escribe su forma factorizada y su cuadrado completado,
#     confirma que las tres formas tienen los mismos coeficientes, y lee las
#     raíces en una y el vértice en la otra.
#     Después factoriza A_ex = [[2, 1, 1], [4, -6, 0], [-2, 7, 2]] una sola vez
#     con scipy.linalg.lu_factor y resuelve A_ex @ x = b para b = [5, -2, 9] y
#     b = [1, 0, 0] reutilizando esa misma factorización.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

# TODO 7 — gcd and divisor count, from the prime factorizations alone.
f360 = {2: 3, 3: 2, 5: 1}
f84 = {2: 2, 3: 1, 7: 1}

shared_ex = {p: min(e, f84[p]) for p, e in f360.items() if p in f84}
gcd_ex = math.prod(p ** e for p, e in shared_ex.items())

print("Shared prime powers / Potencias primas comunes:", shared_ex)
print("gcd from the factors / mcd desde los factores:", gcd_ex,
      "| math.gcd(360, 84):", math.gcd(360, 84))
print("Divisors of 360 / Divisores de 360:",
      math.prod(e + 1 for e in f360.values()),
      "| counted one by one / contados uno a uno:",
      sum(1 for d in range(1, 361) if 360 % d == 0))
print()

# TODO 8 — the quadratic, three ways.
standard_ex = x ** 2 - 6 * x + 8
factored_ex = (x - 2) * (x - 4)
completed_ex = (x - 3) ** 2 - 1

print("Same polynomial / El mismo polinomio:",
      np.allclose(standard_ex.coef, factored_ex.coef),
      np.allclose(standard_ex.coef, completed_ex.coef))
print("(x - 2)(x - 4) -> roots / raíces:", standard_ex.roots())
print("(x - 3)² - 1   -> vertex / vértice: (3, -1)",
      "| check / comprobación: f(3) =", standard_ex(3.0))
print()

# TODO 8 — one LU factorization, two right-hand sides.
A_ex = np.array([[2.0, 1.0, 1.0],
                 [4.0, -6.0, 0.0],
                 [-2.0, 7.0, 2.0]])
factors_ex = lu_factor(A_ex)

for rhs_ex in [np.array([5.0, -2.0, 9.0]), np.array([1.0, 0.0, 0.0])]:
    x_ex = lu_solve(factors_ex, rhs_ex)
    print(f"b = {rhs_ex} -> x = {np.round(x_ex, 6)}",
          "| A @ x == b:", np.allclose(A_ex @ x_ex, rhs_ex))

<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

**The gcd from prime powers.** `360 = 2³ × 3² × 5` and `84 = 2² × 3 × 7`. A common divisor can only use primes both numbers have, and no more copies of each than the poorer of the two owns — so the gcd is `2² × 3 = 12`. `math.gcd` gets the same number by a completely different route (Euclid's algorithm), which is why it is a real check and not a restatement.

**The quadratic.** `(x − 2)(x − 4)` and `(x − 3)² − 1` expand to the same coefficients as `x² − 6x + 8`, so all three *are* one function. The factored form puts the roots `2` and `4` in plain sight; the completed square puts the vertex `(3, −1)` in plain sight. Neither form is "simpler" — each is aimed at a different question.

**One LU, two right-hand sides.** `lu_factor` does the elimination once, `lu_solve` reuses it. That is the whole economic argument for matrix factorizations: the expensive step happens once, and every later solve is cheap. Notebook 12 measures exactly how much that saves.

> 🇪🇸
>
> **El mcd desde las potencias primas.** Un divisor común solo puede usar primos que ambos números tengan, y no más copias de cada uno de las que tenga el más pobre: `2² × 3 = 12`. `math.gcd` llega al mismo número por otro camino (el algoritmo de Euclides), y por eso es una comprobación real.
>
> **La cuadrática.** `(x − 2)(x − 4)` y `(x − 3)² − 1` tienen los mismos coeficientes que `x² − 6x + 8`: son una sola función. La forma factorizada muestra las raíces; el cuadrado completado muestra el vértice. Ninguna es «más simple»; cada una apunta a una pregunta distinta.
>
> **Una LU, dos lados derechos.** `lu_factor` hace la eliminación una vez y `lu_solve` la reutiliza. Ese es todo el argumento económico de las factorizaciones matriciales: el paso caro ocurre una vez. El cuaderno 12 mide cuánto ahorra.

</details>

### Where this goes / Dónde continúa

Two rungs sit above this one, and each has its own deep-dive notebook:

- **[12 · Matrix factorizations](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/12-matrix-factorizations.ipynb)** — LU, Cholesky, QR, eigendecomposition, SVD and NMF side by side, each one made to answer for its cost.
- **[13 · Tensor factorizations](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/13-tensor-factorizations.ipynb)** — the same question one order up: Tucker, CP, Tensor Train and t-SVD, on objects with three or more axes. Section 10 runs Tucker live.

Section 01 only has to establish the move:

**same data, different representation, chosen to make a useful property easier to see.**

> 🇪🇸 Quedan dos peldaños por encima de este, y cada uno tiene su propio cuaderno a fondo:
>
> - **[12 · Factorizaciones matriciales](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/12-matrix-factorizations.ipynb)** — LU, Cholesky, QR, descomposición espectral, SVD y NMF una al lado de otra, y cada una respondiendo por su coste.
> - **[13 · Factorizaciones tensoriales](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/13-tensor-factorizations.ipynb)** — la misma pregunta un orden más arriba: Tucker, CP, Tensor Train y t-SVD, sobre objetos de tres o más ejes. La sección 10 ejecuta Tucker en vivo.
>
> La sección 01 solo tiene que dejar establecido el movimiento:
>
> **los mismos datos, otra representación, elegida para que una propiedad útil sea más fácil de observar.**

## What just happened / Qué acaba de pasar

You now have a practical way to reason about tensors.

### The six ideas to remember / Las seis ideas para recordar

1. **Tensor:** organized numbers / números organizados.
2. **Shape:** how the data is arranged / cómo están organizados los datos.
3. **Axis:** what one direction counts / qué cuenta una dirección.
4. **Unfolding:** rearrange without losing values / reorganizar sin perder valores.
5. **Contraction:** multiply and sum so selected axes disappear / multiplicar y sumar para que algunos ejes desaparezcan.
6. **Factorization:** the same data rewritten so that one question becomes easy / los mismos datos reescritos para que una pregunta se vuelva fácil.

### Final self-check / Autoevaluación final

If you see:

`(1797, 8, 8)`

can you say:

**“1,797 images × 8 pixels high × 8 pixels wide”**

without looking at the answer?

If yes, you are already thinking about tensors in the right way.

> 🇪🇸 Si ves `(1797, 8, 8)`, ¿puedes decir:
>
> **“1.797 imágenes × 8 píxeles de alto × 8 píxeles de ancho”**
>
> sin mirar la respuesta?
>
> Si puedes hacerlo, ya estás razonando sobre tensores de la manera correcta.

---

## Done with this section / Fin de esta sección

Next / Siguiente: **02 · Thinking in N dimensions / Pensar en N dimensiones** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/02-thinking-in-n-dimensions.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)